# 05 — Running the baseline, and making your own submission

This notebook does two things. First it loads the shipped graph-network baseline and scores
it, so you can see exactly what you are up against and how it behaves. Then it walks through
turning your own model into a submission.

**You submit code, not predictions.** A submission is a directory with a `predict.py` in it.
We run it on catalogs you never see, under conditions whose random seeds are not published,
including a simulation code that is not in your training data. That is what makes the test
genuinely a test.

Everything here runs on a CPU in a couple of minutes.

In [ ]:
# Setup. On Colab this installs the toolkit, mounts the data bucket and points the
# environment variables at it. Anywhere else -- a cluster with the data already on disk --
# it does nothing, which is why there is one set of notebooks rather than two.
import sys

if "google.colab" in sys.modules:
    # --force-reinstall, every time, on purpose. Installing only when the package is
    # missing means anyone who ran a notebook once keeps a stale copy forever, and during
    # an event where fixes are being pushed that is exactly backwards. --no-deps keeps it
    # to a few seconds: everything it depends on is already in the runtime.
    %pip install -q --upgrade --force-reinstall --no-deps git+https://github.com/xwzhang98/kaai-robust-inference-hackathon-2026
    # Drop anything already imported from the old copy, so this works without a restart.
    for _name in [m for m in sys.modules if m.startswith("kaai_hackathon")]:
        del sys.modules[_name]

from kaai_hackathon.colab_setup import setup

setup()


## Setup

In [ ]:
%matplotlib inline
import os
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from kaai_hackathon import PUBLIC_SUITES
from kaai_hackathon.scoring import leaderboard_table, macro_average, r2
from kaai_hackathon.splits import example_sims, load_labels, local_split
from kaai_hackathon.submission import REQUIRED_KEYS, load_submission, validate_prediction

DATA_ROOT = Path(os.environ["CAMELS_HACKATHON_DATA"])
PARAMS_ROOT = Path(os.environ.get("CAMELS_HACKATHON_PARAMS", DATA_ROOT))
BASELINE = Path("..") / "baseline"
TARGETS = ("Omega_m", "sigma_8")


def catalog_path(suite, sim_id):
    return DATA_ROOT / suite / f"LH_{sim_id}" / "groups_090.hdf5"

## 1. The contract

Two functions. That is the whole interface.

```python
def load_model(model_dir: str) -> object:
    # Called once. Put anything expensive here.
    ...

def predict(model, catalog_path: str) -> dict:
    # Called once per catalog. Return {"Omega_m": float, "sigma_8": float},
    # and optionally the four feedback parameters for the bonus track.
    ...
```

`load_model` gets the path to your own submission directory, so that is where your weights
live. `predict` gets a path to one catalog file and nothing else.

Here is the whole of the shipped graph baseline's `predict.py`. It is short on purpose.

In [ ]:
print((BASELINE / "gnn" / "predict.py").read_text())

## 2. Loading it the way the scorer does

`load_submission` is the same function the organizers' evaluation runner calls. If your
submission works here, it works there.

In [ ]:
started = time.time()
model, predict = load_submission(BASELINE / "gnn")
print(f"load_model took {time.time() - started:.2f}s")

example = catalog_path("IllustrisTNG", local_split("IllustrisTNG")["test"][0])
predict(model, str(example))                      # warm-up: the first call imports and
                                                  # allocates, and is not representative
started = time.time()
prediction = predict(model, str(example))
print(f"predict took {time.time() - started:.3f}s on one catalog")
print(prediction)
print("\nvalidated:", validate_prediction(prediction))
print("required keys:", REQUIRED_KEYS)

`validate_prediction` is worth knowing about, because it is what stands between a small bug
in your model and a mystifying leaderboard row. It rejects a missing key, a non-finite value,
and anything that is not a number, and it does it on the catalog that produced the problem
rather than three hundred catalogs later.

## 3. Scoring it on catalogs it has never seen

The 100 simulations per suite that we hold out are not available to you, so this uses the
`private_test` ids purely as *a* held-out set — the model was trained on the `public` ids and
has never seen these, which is the property that matters here.

When you score your own model, hold out your own simulations, and **hold them out by
simulation id**, never by catalog row or by galaxy. Two galaxies from the same simulation are
not independent samples.

In [ ]:
N_PER_SUITE = 60

started = time.time()
predictions, truths, suites = [], [], []
for suite in PUBLIC_SUITES:
    labels = load_labels(PARAMS_ROOT, suite)
    for sim_id in local_split(suite)["test"][:N_PER_SUITE]:
        out = validate_prediction(predict(model, str(catalog_path(suite, sim_id))))
        predictions.append([out[t] for t in TARGETS])
        truths.append(labels[sim_id, :2])
        suites.append(suite)

predictions = np.asarray(predictions)
truths = np.asarray(truths)
suites = np.asarray(suites)
elapsed = time.time() - started
print(f"{len(truths)} catalogs in {elapsed:.1f}s  ({elapsed / len(truths) * 1000:.0f} ms each)")

In [ ]:
per_suite = {suite: {t: r2(truths[suites == suite][:, j], predictions[suites == suite][:, j])
                     for j, t in enumerate(TARGETS)}
             for suite in PUBLIC_SUITES}
macro = macro_average(per_suite, TARGETS)

print(f"{'':16s} {'Omega_m':>9s} {'sigma_8':>9s}")
for suite in PUBLIC_SUITES:
    print(f"{suite:16s} " + " ".join(f"{per_suite[suite][t]:9.3f}" for t in TARGETS))
print(f"{'macro average':16s} " + " ".join(f"{macro[t]:9.3f}" for t in TARGETS))

Suites are macro-averaged — every simulation code counts the same regardless of how many
catalogs it contributes — and `Omega_m` and `sigma_8` are never folded into one number.
`sigma_8` is the hard one and hiding it inside an average would be the wrong kind of kindness.

Now look at the predictions rather than the summary. A single $R^2$ hides almost everything
interesting about how a model fails.

In [ ]:
PRIOR = {"Omega_m": (0.1, 0.5), "sigma_8": (0.6, 1.0)}
COLOURS = dict(zip(PUBLIC_SUITES, ("C0", "C1", "C2")))

fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.6))
for j, (ax, target) in enumerate(zip(axes, TARGETS)):
    lo, hi = PRIOR[target]
    for suite in PUBLIC_SUITES:
        m = suites == suite
        ax.scatter(truths[m, j], predictions[m, j], s=22, alpha=0.75,
                   color=COLOURS[suite],
                   label=f"{suite}  $R^2$ = {per_suite[suite][target]:+.3f}")
    ax.plot([lo, hi], [lo, hi], "k--", lw=1.2, zorder=0)
    ax.axhline(np.mean(PRIOR[target]), color="0.65", lw=1, ls=":", zorder=0)
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi); ax.set_aspect("equal")
    ax.set_xlabel(f"true {target}"); ax.set_ylabel(f"predicted {target}")
    ax.set_title(f"{target}    macro $R^2$ = {macro[target]:+.3f}")
    ax.legend(loc="upper left", fontsize=9, framealpha=0.9)
fig.suptitle("Shipped GNN baseline on held-out simulations", y=1.02)
plt.tight_layout()

The dashed diagonal is a perfect prediction. The dotted horizontal line is the centre of the
prior — a model that has learned nothing sits on it, and scores $R^2 = 0$.

Two things to read off these panels.

**$\Omega_m$ tracks the diagonal across the whole range**, and the three suites lie on top of
each other. That is a model that has learned something transferable.

**$\sigma_8$ is compressed towards the prior mean.** The points form a band that is flatter
than the diagonal: the model moves in the right direction but not far enough, hedging towards
the middle because it is uncertain. That is what a regression looks like when it has a weak
signal, and it is the honest state of the art for $\sigma_8$ from a 25 Mpc$/h$ catalog rather
than a defect of this particular model.

If your own $\sigma_8$ panel ever looks like the $\Omega_m$ one, you have done something
genuinely new — check it hard before you believe it.

## 4. Scoring under the conditions, locally

The leaderboard is a table of (condition x target), not a number. You can build your own copy
of it with the same registry the scorer uses, and you should — a model that is excellent on
clean catalogs and falls apart under mild noise will look fine in your own validation and bad
in ours.

In [ ]:
from kaai_hackathon.catalog_io import read_catalog, write_catalog
from kaai_hackathon.conditions import (
    PUBLISHED_CONDITIONS, apply_condition, condition_seed,
)

# Enough columns for every condition to be applied faithfully.
GROUP_FIELDS = ["GroupPos", "GroupCM", "GroupVel", "GroupMass",
                "GroupNsubs", "GroupFirstSub"]
SUBHALO_FIELDS = ["SubhaloPos", "SubhaloCM", "SubhaloVel", "SubhaloSpin",
                  "SubhaloMass", "SubhaloMassType", "SubhaloGrNr", "SubhaloParent"]

MY_SEED = 12345          # yours, not the organizers'. Theirs is not published.
N_SHIFT = 30

import tempfile

started = time.time()
scores = {}
for spec in PUBLISHED_CONDITIONS:
    got, want = [], []
    for suite in PUBLIC_SUITES:
        labels = load_labels(PARAMS_ROOT, suite)
        for sim_id in local_split(suite)["test"][:N_SHIFT]:
            cat = read_catalog(catalog_path(suite, sim_id), group_fields=GROUP_FIELDS,
                               subhalo_fields=SUBHALO_FIELDS)
            shifted = apply_condition(cat, spec,
                                      condition_seed(MY_SEED, spec.name, suite, sim_id))
            # The scorer hands your code a FILE, so score yourself the same way.
            with tempfile.TemporaryDirectory() as scratch:
                path = Path(scratch) / "groups_090.hdf5"
                write_catalog(shifted, path)
                out = validate_prediction(predict(model, str(path)))
            got.append([out[t] for t in TARGETS])
            want.append(labels[sim_id, :2])
    got, want = np.asarray(got), np.asarray(want)
    scores[spec.name] = {t: r2(want[:, j], got[:, j]) for j, t in enumerate(TARGETS)}
    print(f"{spec.name:14s} {spec.tier:5s} " +
          "  ".join(f"{t} {scores[spec.name][t]:+.3f}" for t in TARGETS), flush=True)
print(f"\n[{time.time() - started:.0f}s]")

In [ ]:
print(leaderboard_table({"gnn_baseline": scores}))

That is the shape of the leaderboard: one row per team, one column per (condition, target).
You can win a column without winning the table.

Note what the conditions cost this particular model — the symmetry rows barely move, the
position-noise rows hurt, and `vel_noise` hurts `Omega_m` specifically, because this model
reads velocities. A model that ignored velocities would show `vel_noise` as a flat zero, and
that would be a statement about the model, not about its robustness.

Two caveats on that table. Your seed is not our seed, so the draws differ even though the
recipes are identical. And it is built on 30 simulations per suite, where an $R^2$ carries a
scatter of a few hundredths — enough that a symmetry row can come out slightly *above* clean.
Raise `N_SHIFT` before reading anything into a small difference.

## 5. Writing your own

Copy a baseline directory and edit it. Here is the smallest possible submission that is not
the prior mean — it fits nothing, it just reads a column — written out and scored end to end
so you can see the whole loop.

In [ ]:
from kaai_hackathon.catalog_io import read_catalog

MSTAR_CUT = 1.95e-2          # the same cut the shipped baseline uses, in 1e10 Msun/h


def galaxy_count(path):
    cat = read_catalog(path, group_fields=[], subhalo_fields=["SubhaloMassType"])
    return int((cat.subhalo["SubhaloMassType"][:, 4] > MSTAR_CUT).sum())


# Fit one straight line, Omega_m against log10(N), on catalogs the model may look at.
counts, omegas = [], []
for suite in PUBLIC_SUITES:
    labels = load_labels(PARAMS_ROOT, suite)
    for sim_id in local_split(suite)["train"][:120]:
        counts.append(np.log10(max(galaxy_count(catalog_path(suite, sim_id)), 1)))
        omegas.append(labels[sim_id, 0])
slope, intercept = np.polyfit(counts, omegas, 1)
print(f"Omega_m ~ {slope:.4f} * log10(N) + {intercept:.4f}   "
      f"(fitted on {len(counts)} public catalogs)")

MY_SUBMISSION = Path("my_submission")
MY_SUBMISSION.mkdir(exist_ok=True)
(MY_SUBMISSION / "predict.py").write_text(f'''
# The smallest submission that is not the prior mean: one straight line through the
# galaxy count. It exists to show the shape of a real submission, and to be a floor.
import numpy as np

from kaai_hackathon.catalog_io import read_catalog


def load_model(model_dir):
    # Anything expensive belongs here -- unpickling, torch.load, moving to the GPU.
    # It runs once for the whole evaluation, not once per catalog.
    return {{"slope": {slope!r}, "intercept": {intercept!r}}}


def predict(model, catalog_path):
    cat = read_catalog(catalog_path, group_fields=[],
                       subhalo_fields=["SubhaloMassType"])
    n_galaxies = int((cat.subhalo["SubhaloMassType"][:, 4] > {MSTAR_CUT!r}).sum())
    return {{"Omega_m": float(model["slope"] * np.log10(max(n_galaxies, 1))
                              + model["intercept"]),
            "sigma_8": 0.8}}            # a constant: R^2 = 0 by construction
''')

my_model, my_predict = load_submission(MY_SUBMISSION)
got = []
for suite in PUBLIC_SUITES:
    for sim_id in local_split(suite)["test"][:N_PER_SUITE]:
        out = validate_prediction(my_predict(my_model, str(catalog_path(suite, sim_id))))
        got.append([out[t] for t in TARGETS])
got = np.asarray(got)

print(f"{'':22s} {'Omega_m':>9s} {'sigma_8':>9s}")
for name, values in (("count-only submission", got), ("shipped GNN baseline", predictions)):
    print(f"{name:22s} " +
          " ".join(f"{r2(truths[:, j], values[:, j]):9.3f}" for j in range(2)))

One straight line through the galaxy count gets a real fraction of the way on $\Omega_m$,
from a model with two parameters that never looks at where anything is. On $\sigma_8$ it
returns a constant, which is what $R^2 \approx 0$ means and is the honest floor.

Keep that number in mind. If a model of yours beats it on $\Omega_m$, the useful question is
how much of the gain came from counting galaxies better rather than from reading their
arrangement. Notebook 04's baseline is handed $\log_{10} N$ as an explicit input for exactly
this reason — the published model treats abundance as information rather than pretending it
is not there, and so should your reading of any score.

## 6. Before you submit

A checklist, in the order things actually go wrong.

1. **It loads from a clean process.** `load_submission("your_dir")` in a fresh kernel. Most
   failures are an import that only worked because your notebook had already run something.
2. **It runs in the provided environment**, with no `pip install` at evaluation time. If you
   need a package that is not there, ask before the deadline, not at it.
3. **It reads only the catalog it is given, plus its own directory.** No network at
   evaluation time.
4. **It stays under the per-catalog wall-clock cap.** The shipped GNN takes about a tenth of a
   second per catalog on a CPU. If yours takes ten seconds, say so early.
5. **It never returns a NaN.** Catalogs at the low-$\Omega_m$ corner can have very few
   galaxies above a mass cut, and a model that divides by the count will produce one. Test on
   the sparsest catalogs you can find, not the average ones.
6. **It does not read `Header`, `Parameters` or `Config`.** Those are stripped from every
   catalog your code is evaluated on, so it is pointless as well as against the rules.

There is a **mandatory smoke submission at the halfway mark**. It is not graded. It exists
because the most common way to lose a hackathon is to discover at the deadline that your code
does not run on someone else's machine, and the fix takes ten minutes if you find out with
hours to spare.

In [ ]:
# The five-second version of that checklist.
import subprocess
import sys

check = subprocess.run(
    [sys.executable, "-c",
     "from kaai_hackathon.submission import load_submission, validate_prediction; "
     f"m, p = load_submission({str(MY_SUBMISSION.resolve())!r}); "
     f"print(validate_prediction(p(m, {str(example)!r})))"],
    capture_output=True, text=True)
print(check.stdout or check.stderr)

## What you should take away

1. A submission is a directory with `predict.py` exposing `load_model` and `predict`.
2. Score yourself with the same registry the organizers use, on a split you made **by
   simulation id**, under every condition — not just clean.
3. Look at the scatter, not only the $R^2$. A compressed $\sigma_8$ band and a genuinely bad
   model score about the same and mean completely different things.
4. Test the sparse catalogs and the failure cases. A single NaN takes down a whole condition.
5. Send the halfway smoke submission.